In [1]:
# !rm -rf logs/ # clear logs
# !rm -rf optimizer_output/

# Imports

In [2]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice again
!ngspice -v

PDK_ROOT: /foss/pdks
SPICE_USERINIT_DIR: /foss/pdks/ihp-sg13g2/libs.tech/ngspice
******
** ngspice-44.2 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2024, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Sat May 24 09:38:33 UTC 2025
******


In [3]:
import sympy as sp
import logging

from pathlib import Path

from symxplorer.spice_engine             import Spicelib_Wrapper, Sim_Execution_Type
from symxplorer.designer_tools           import Nevergrad_Spice_Multi_Spec_Optimizer, Project_Setup

from symxplorer.logging import setup_loggers

logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

2025-09-30 06:24:47,687 - SymXplorer.optimizer - Using device: cpu and dtype: torch.float64
2025-09-30 06:24:47,698 - SymXplorer.jupyter - Spicelib_Wrapper imported successfully.


# Instantiations


In [4]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

06:24:47 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
06:24:47 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/logs/SymXplorer_2025-09-30_06-24-47.log
06:24:47 - SymXplorer: [INFO] 🔧 spicelib logger set to 50


In [5]:
# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

06:24:47 - SymXplorer.domains: [INFO] 📂 Loading project setup from /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml
06:24:48 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: NGOpt, type=nevergrad, budget=100, random_seed=48
06:24:48 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
06:24:48 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
06:24:48 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
06:24:48 - SymXplorer.domains: [INFO] 	Number of target specs: 1
06:24:48 - SymXplorer.domains: [INFO] 		- TargetSpec(name=fc, target=1e6, tolerance=1000.0, goal=OptimizationGoalType.EXACT, sim_type=SimType.AC, enable=True)
06:24:48 - SymXplorer.domains: [INFO] Project 'Tunable-TIA' initialized with simulator 'ngspice'
06:24:48 - SymXplorer.domains: [INFO] 	Workspace root: /foss/designs/eda/SymXplorer
06:24:48 - SymXplorer.dom

Project_Setup(name='Tunable-TIA', description='Tunable TIA BPF example sizing in the ihp-sg13g2 technology', simulator='ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer'), netlist=PosixPath('examples/tunable-tia/ihp-sg13g2/spice/tb_ac.spice'), outdir=PosixPath('examples/tunable-tia/scripts/optimizer_output'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07), 'max_cap_w': np.float64(9.999999999999999e-06), 'min_cap_w': np.float64(1e-06), 'max_cap_l': np.float64(9.999999999999999e-06), 'min_cap_l': np.float64(1e-06), 'max_res_w': np.float64(9.999999999999999e-06), 'min_res_w': np.float64(1e-06), 'max_res_l': np.float64(9.999999999999999e-06), 'min_

In [6]:
# (2) Create the Spice Simulator Wrapper
wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.name,
    netlist_filename= PROJECT_SETUP.ws_root / PROJECT_SETUP.netlist,
    output_folder=PROJECT_SETUP.ws_root / PROJECT_SETUP.outdir,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

06:24:48 - SymXplorer.spicelib: [WARNING] ⚠️ Output directory already exists, re-creating: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
06:24:48 - SymXplorer.spicelib: [INFO] --------------------------------------------------
06:24:48 - SymXplorer.spicelib: [INFO] 🚀 Spicelib_Wrapper initialized successfully!
06:24:48 - SymXplorer.spicelib: [INFO] 	📝 Project: Tunable-TIA
06:24:48 - SymXplorer.spicelib: [INFO] 	📜 Schematic: tb_ac
06:24:48 - SymXplorer.spicelib: [INFO] 	📂 Output Folder: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
06:24:48 - SymXplorer.spicelib: [INFO] --------------------------------------------------
06:24:48 - SymXplorer.spicelib: [INFO] Using ngspice from ['/foss/tools/bin/ngspice']
06:24:48 - SymXplorer.spicelib: [INFO] 📊 --- Circuit Information ---
06:24:48 - SymXplorer.spicelib: [INFO] 🔗 Nodes in the netlist: ['VSS', 'GND', 'VDD', 'Vbias', 'Von', 'Vop', 'In', 'Ip']
06:24:48 - SymXplorer.spicelib: [INFO] Te

In [7]:
circuit_optimizer = Nevergrad_Spice_Multi_Spec_Optimizer(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

# Sanity Check

In [8]:
wrapper.run_sanity_check(
    use_editor=True,
    sim_execution_t=Sim_Execution_Type.RUN_NOW
)

06:24:48 - SymXplorer.spicelib: [INFO] 📂 Creating dedicated sanity check folder...
06:24:48 - SymXplorer.spicelib: [INFO] 🧪 Running sanity check simulation...
06:24:49 - SymXplorer.spicelib: [INFO] simulator log: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/sanity_check/Tunable-TIA_sanity.log
06:24:49 - SymXplorer.spicelib: [INFO] simulator RAW: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/sanity_check/Tunable-TIA_sanity.raw
06:24:49 - SymXplorer.spicelib: [INFO] 🔎 Verifying simulation results...
06:24:49 - SymXplorer.spicelib: [INFO] ✅ Sanity check passed 🎉


True

# Method Calls

In [9]:
circuit_optimizer.parameterize()

Dict(x_dut_cap_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_cap_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_w=Log{Cl(0,6,b),exp=2.15},x_dut_res_3_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_3_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}]):{'x_dut_nfet_w': 10.000000000000002, 'x_dut_nfet_l': 50.0, 'x_dut_cap_w': 50.0, 'x_dut_cap_l': 50.0, 'x_dut_res_s_l': 50.0, 'x_dut_res_s_w': 50.0, 'x_dut_res_3_l': 50.0, 'x_dut_res_3_w': 50.0}

In [10]:
circuit_optimizer.optimize()

06:25:03 - SymXplorer.optimizer: [INFO] Optimizer is set to NGOpt with budget = 100
Optimizing:   0%|          | 0/100 [00:00<?, ?trial/s]2025-09-30 06:25:03,863 - nevergrad.optimization.optimizerlib - NGOpt16 selected Cobyla optimizer.
2025-09-30 06:25:03,866 - nevergrad.optimization.optimizerlib - NGOpt selected Cobyla optimizer.
Optimizing:   4%|▍         | 4/100 [01:17<30:49, 19.26s/trial]  


IndexError: <spicelib.raw.raw_read.PlotData object at 0xffff261d57c0> doesn't contain trace "fc"
Valid traces are ['frequency', '@n.x_dut.xm1.nsg13_lv_nmos[cdd]', '@n.x_dut.xm1.nsg13_lv_nmos[cgb]', '@n.x_dut.xm1.nsg13_lv_nmos[cgd]', '@n.x_dut.xm1.nsg13_lv_nmos[cgdol]', '@n.x_dut.xm1.nsg13_lv_nmos[cgg]', '@n.x_dut.xm1.nsg13_lv_nmos[cgs]', '@n.x_dut.xm1.nsg13_lv_nmos[cgsol]', '@n.x_dut.xm1.nsg13_lv_nmos[cjd]', '@n.x_dut.xm1.nsg13_lv_nmos[cjs]', '@n.x_dut.xm1.nsg13_lv_nmos[css]', '@n.x_dut.xm1.nsg13_lv_nmos[gds]', '@n.x_dut.xm1.nsg13_lv_nmos[gm]', '@n.x_dut.xm1.nsg13_lv_nmos[gmb]', 'i(@n.x_dut.xm1.nsg13_lv_nmos[ids])', 'v(@n.x_dut.xm1.nsg13_lv_nmos[l])', 'v(@n.x_dut.xm1.nsg13_lv_nmos[sfl])', 'v(@n.x_dut.xm1.nsg13_lv_nmos[sid])', 'v(@n.x_dut.xm1.nsg13_lv_nmos[vds])', 'v(@n.x_dut.xm1.nsg13_lv_nmos[vdss])', 'v(@n.x_dut.xm1.nsg13_lv_nmos[vgs])', 'v(@n.x_dut.xm1.nsg13_lv_nmos[vth])', 'flo', 'gain_db', 'gmax', 'gmax_3db', 'v(in)', 'v(ip)', 'i(l.x_dut.l2)', 'i(l.x_dut.l3)', 'v(n.x_dut.xm1.nsg13_

In [ ]:
circuit_optimizer.plot_loss(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

In [ ]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss = out

In [ ]:
circuit_optimizer.plot_solution(best_param)